# Lab 1: First steps with PyTorch and neural networks
**Contemporary Machine Learning in Theory and Practice · WS26/27**

**Tutorial:** 22 October 2026 · 90 minutes  
**Homework deadline:** 29 October 2026 · Groups of at most four  
**Group name:**

We study a small regression problem:
\[
X\sim\operatorname{Unif}[-1,1],\qquad Y=X^2+\varepsilon,
\qquad \varepsilon\sim N(0,0.1^2),
\]
with $X$ and $\varepsilon$ independent. We will represent observations as
tensors, construct a small neural network, measure its error, and run a supplied
training program. No previous NumPy or PyTorch knowledge is assumed.

| Exercise | Time |
|---|---:|
| 1. Tensors and basic operations | 20 min |
| 2. Generate regression data | 15 min |
| 3. Construct a small neural network | 25 min |
| 4. Measure prediction error | 10 min |
| 5. Run the supplied training program | 20 min |

Complete the setup check (`setup/check_setup.ipynb` in your course checkout) before class. Work in a
personal copy, enter your group name, and save regularly. Everything here runs
on the CPU. You need PyTorch and Matplotlib; no separate helper file is needed.

**How to work with this notebook.** Run cells from top to bottom with
**Shift+Enter**. A code cell runs Python; a text cell contains explanations or your
written answer. Cells labelled **Supplied** are ready to run. In **Your turn**
cells, replace `...` or add the requested short lines. The dots are placeholders,
not working answers: dependent cells will fail until you fill them in.
Predict an output on paper, try the code, and discuss the result with your group.

The plotting code and parameter updates are supplied. You are responsible for
the tensor operations, the network's predictions, its loss, and interpreting the
results. We will study model classes and training machinery in Labs 2 and 3.

In [ ]:
# Supplied: import the two libraries used in this notebook.
import torch
import matplotlib.pyplot as plt

# The small computations in this lab use the CPU and one CPU thread.
torch.set_num_threads(1)
print("PyTorch version:", torch.__version__)

## 1. Tensors and basic operations · 20 minutes

A **tensor** stores a number, vector, matrix, or an array with more dimensions.
Run this first example. The decimal points specify floating-point numbers.
Python starts counting entries at zero, and `**` denotes a power.

In [ ]:
# Supplied example.
v = torch.tensor([-1., 0., 2.])
print("v:", v)
print("shape:", v.shape)
print("first entry:", v[0])
print("squares:", v**2)

Use this small reference as needed:

| Meaning | PyTorch expression |
|---|---|
| Create a vector | `torch.tensor([1., 2., 3.])` |
| Inspect its shape | `v.shape` |
| First entry / first three entries | `v[0]` / `v[:3]` |
| Sum / mean | `v.sum()` / `v.mean()` |
| Square each entry | `v**2` |
| Multiply corresponding entries | `A * B` |
| Matrix multiplication | `A @ B` |
| Transpose a matrix | `A.T` |
| Replace negative entries by zero | `torch.relu(v)` |

**Your turn (a).** Create $u=(-2,-1,0,1,2)$. Compute its sum, its mean square,
and its first three entries. Check the results by hand.

In [ ]:
# Your turn: create u and add three print commands.
u = ...

**Your turn (b).** The function $\operatorname{ReLU}(t)=\max\{t,0\}$ acts on
each entry of a tensor. Predict `torch.relu(u)`, then run it. Change one entry
of `u` and repeat. You can assign a new value with, for example, `u[0] = 3.`.

In [ ]:
# Your turn: print the positive part, change one entry, and print it again.

**Your turn (c).** Let
\[
A=\begin{pmatrix}1&2\\0&1\end{pmatrix},\qquad
B=\begin{pmatrix}1&0\\3&2\end{pmatrix}.
\]
Predict the results of `A * B` and `A @ B`, then compare them in PyTorch.
Explain the difference between the operations.

In [ ]:
# Supplied matrices; add the two products and print them.
A = torch.tensor([[1., 2.], [0., 1.]])
B = torch.tensor([[1., 0.], [3., 2.]])
# Your turn below.

*Write your calculation or observations here.*

**Your turn (d).** Run the supplied cell and compare the two shapes.
`reshape(-1, 1)` arranges entries in one column; `-1` asks PyTorch to work out
the number of rows. The entries have not changed, but their arrangement has.

Throughout the regression exercises, $n$ scalar observations will have shape
**`(n, 1)`**: one row per observation and one column per scalar input or target.

In [ ]:
# Supplied: restore the original vector, then view it as a column.
u = torch.tensor([-2., -1., 0., 1., 2.])
column = u.reshape(-1, 1)
print("vector shape:", u.shape)
print("column shape:", column.shape)
print(column)

*Write your calculation or observations here.*

## 2. Generate regression data · 15 minutes

`torch.rand(n, 1)` creates an $(n,1)$ tensor with uniform entries in $[0,1)$.
`torch.randn(n, 1)` creates standard normal entries. A **seed** makes a sequence
of random draws reproducible.

**Your turn (a).** Generate $n=64$ observations of our regression problem.
Fill the three expressions below. **Hint:** if $U$ is uniform on $[0,1)$,
then $2U-1$ is uniform on $[-1,1)$. Multiplying a standard normal variable by
$0.1$ gives noise with standard deviation $0.1$.

In [ ]:
# Your turn: fill in these three expressions.
torch.manual_seed(7)
n = 64
x = ...
noise = ...
y = ...

# Supplied inspection commands.
print("input and target shapes:", x.shape, y.shape)
print("first five inputs:\n", x[:5])
print("first five targets:\n", y[:5])

**Supplied plot.** Here `grid` contains equally spaced inputs. The expression
`x[:, 0]` selects all rows (`:`) in the first column (`0`) for plotting.
Run this cell once the data cell is complete.

In [ ]:
grid = torch.linspace(-1, 1, 200).reshape(-1, 1)
plt.figure(figsize=(7, 4))
plt.scatter(x[:, 0], y[:, 0], label="Observations", s=22)
plt.plot(grid[:, 0], grid[:, 0]**2, color="black", label="True regression function")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.tight_layout()
plt.show()

**Your turn (b).** In the data cell, change the noise standard deviation to
$0$, rerun that cell and the plot, and then try $0.3$. What changes? Which
function describes the centre of the observations?

**Before continuing:** restore the standard deviation to **$0.1$**, then
rerun the complete data cell and its plot. Reusing seed 7 in the complete data
cell keeps the inputs and underlying normal draws the same across these trials.

*Write your calculation or observations here.*

## 3. Construct a small neural network · 25 minutes

A network with one hidden layer and two hidden neurons has the form
\[
z=W_1x+b_1,\qquad h=\operatorname{ReLU}(z),\qquad
f_\theta(x)=W_2h+b_2.
\]
Begin with
\[
W_1=\begin{pmatrix}1\\-1\end{pmatrix},\quad
b_1=\begin{pmatrix}-0.5\\-0.5\end{pmatrix},\quad
W_2=\begin{pmatrix}2&-1\end{pmatrix},\quad b_2=0.5.
\]

**Your turn (a), on paper.** Calculate $z$, $h$, and $f_\theta(1)$.
Which hidden neuron is active (has a strictly positive output)?

*Write your calculation or observations here.*

In [ ]:
# Supplied: the four parameter tensors for the hand calculation.
W1 = torch.tensor([[1.], [-1.]])
b1 = torch.tensor([-0.5, -0.5])
W2 = torch.tensor([[2., -1.]])
b2 = torch.tensor([0.5])

**Your turn (b).** We store a batch of $n$ scalar inputs as an $(n,1)$ tensor
`x`. Each observation is a row, so the same formulas become
\[
Z=xW_1^{\mathsf T}+b_1,\qquad H=\operatorname{ReLU}(Z),\qquad
\widehat Y=HW_2^{\mathsf T}+b_2.
\]
PyTorch adds the bias vector to **every row**. Thus `Z` and `H` have shape
`(n, 2)`, and predictions have shape `(n, 1)`.

Fill the three expressions below. `def predict(x):` defines a function called
`predict` with input `x`; the indented lines form its body, and `return` gives its
output. Calling `predict(x)` reuses these lines. In this first lab the function
uses the currently defined tensors `W1`, `b1`, `W2`, and `b2`.

In [ ]:
# Your turn: translate the three displayed formulas into tensor operations.
def predict(x):
    z = ...
    h = ...
    prediction = ...
    return prediction

In [ ]:
# Supplied checks: compare the first output with your hand calculation.
one_input = torch.tensor([[1.]])
small_batch = torch.tensor([[-1.], [0.], [1.]])
print("prediction at 1:", predict(one_input))
print("three predictions:\n", predict(small_batch))
print("hidden shape:", (small_batch @ W1.T + b1).shape)
print("prediction shape:", predict(small_batch).shape)

**Your turn (c).** Run the supplied plot. Change the first entry of `b1` from
$-0.5$ to $0$ in the parameter cell, rerun that cell, and rerun the plot.
Which corner of the graph moves? Explain using $\operatorname{ReLU}(x+b)$.
Then restore the original bias and rerun both cells.

In [ ]:
# Supplied plot of the current two-neuron network.
plt.figure(figsize=(7, 4))
plt.plot(grid[:, 0], predict(grid)[:, 0].detach(), label="Current network")
plt.xlabel("x")
plt.ylabel("prediction")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

**Your turn (d).** Temporarily replace `h = torch.relu(z)` by `h = z` in your
function, rerun the function cell, and rerun the plot. Why does the graph become
a straight line even though there are still two layers?

**Before continuing:** restore `h = torch.relu(z)` and rerun the function cell.
The supplied reset below restores the original two-neuron parameters and checks
your function at three inputs. It does not replace your function.

*Plotting note:* `.detach()` gives values without derivative tracking so that
Matplotlib can draw them. This becomes useful when our parameters are trainable.

*Write your calculation or observations here.*

In [ ]:
# Supplied reset/check. Keep the ReLU in your predict function.
W1 = torch.tensor([[1.], [-1.]])
b1 = torch.tensor([-0.5, -0.5])
W2 = torch.tensor([[2., -1.]])
b2 = torch.tensor([0.5])
assert torch.allclose(predict(small_batch), torch.tensor([[0.], [0.5], [1.5]])),     "Check predict, and restore h = torch.relu(z) before continuing."
print("The predictions agree with the hand calculation.")

## 4. Measure prediction error · 10 minutes

For predictions $\widehat y_i$ and observations $y_i$, the **mean squared error** is
\[
L=\frac{1}{n}\sum_{i=1}^n(\widehat y_i-y_i)^2.
\]

**Your turn (a).** For predictions $(2,1)$ and targets $(1,3)$, first compute
$L$ by hand. Then calculate the same value with the supplied tensors below.
Use subtraction, `**2`, and `.mean()`.

In [ ]:
prediction_small = torch.tensor([2., 1.])
target_small = torch.tensor([1., 3.])
# Your turn.
loss_small = ...
print(loss_small)

**Your turn (b).** Compute the network's predictions for the 64 observations and
complete the loss expression. Check that predictions and targets have the same
shape **before subtracting** them. A column `(n, 1)` and a vector `(n,)` can
combine into a matrix in PyTorch, so matching shapes matters.

In [ ]:
# Supplied predictions and shape check; complete the loss.
prediction = predict(x)
print("prediction and target shapes:", prediction.shape, y.shape)
assert prediction.shape == y.shape
loss = ...
print("Mean squared error:", loss.item())

**Your turn (c).** Change `b2` in the parameter cell, rerun that cell and the
loss cell, and compare the total loss. Does improving the prediction at one
observation necessarily improve the average loss? Explain.

Restore `b2 = torch.tensor([0.5])` when you finish. The next exercise will start
from a fresh network, so your experiments will not affect its initialization.

*Write your calculation or observations here.*

## 5. Run the supplied training program · 20 minutes

Training changes the network's parameters to reduce the loss on the observed
data. We now use **16 hidden neurons** so that the same formula can produce a
more flexible piecewise affine function. Your `predict` function already works
for this width: none of its three expressions needs to change.

**Supplied initialization.** The following cell creates a fresh network and
optimizer. `requires_grad=True` asks PyTorch to track how the loss depends on a
parameter. The hidden parameters start randomly; the output weights and bias
start at zero. This makes the initial prediction zero without making all hidden
neurons identical. The optimizer will update all four parameter tensors.

**For a fresh training run, rerun the initialization cell and then the training
cell.** Rerunning only the training cell continues training the existing model.
The observations `x` and `y` remain the restored data from Exercise 2.

In [ ]:
# Supplied: reset the network, optimizer, and saved results.
torch.manual_seed(11)
width = 16
W1 = torch.randn(width, 1, requires_grad=True)
b1 = torch.randn(width, requires_grad=True)
W2 = torch.zeros(1, width, requires_grad=True)
b2 = torch.zeros(1, requires_grad=True)
optimizer = torch.optim.Adam([W1, b1, W2, b2], lr=0.02)
number_of_steps = 2000

before_training = predict(grid).detach().clone()
loss_history = []
print("Hidden width:", width)
print("Initial training MSE:", ((predict(x) - y)**2).mean().item())

**Your turn (a).** Fill **only the prediction and loss expressions** below,
using what you wrote in Exercises 3 and 4. All update commands are supplied.

- `optimizer.zero_grad()` clears the previous derivative information.
- `loss.backward()` calculates derivatives of the current loss.
- `optimizer.step()` uses them to change the parameters.

The loop repeats these steps. You do not need to implement the optimizer or
explain its update formula today; we will study this in Lab 3. The final line
inside the loop simply saves a numerical loss for the plot.

In [ ]:
# Supplied loop: fill only the two familiar expressions marked below.
for step in range(number_of_steps):
    optimizer.zero_grad()
    prediction = ...                       # Your turn: call your function.
    loss = ...                             # Your turn: mean squared error.
    loss.backward()
    optimizer.step()
    loss_history.append(loss.item())

# Supplied: include the loss after the final update.
with torch.no_grad():
    final_loss = ((predict(x) - y)**2).mean().item()
loss_history.append(final_loss)
print("Final training MSE:", final_loss)

**Supplied results.** Run the next two cells. `torch.no_grad()` is used when
we only want to evaluate the trained network, without tracking derivatives.
The plotting and evaluation code is provided in full.

In [ ]:
# Supplied: compare predictions before and after training, and show the loss.
with torch.no_grad():
    after_training = predict(grid)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(x[:, 0], y[:, 0], s=18, alpha=0.7, label="Training observations")
axes[0].plot(grid[:, 0], grid[:, 0]**2, "k", label="True function: x²")
axes[0].plot(grid[:, 0], before_training[:, 0], "--", label="Before training")
axes[0].plot(grid[:, 0], after_training[:, 0], label="After training")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y / prediction")
axes[0].legend()
axes[1].plot(loss_history)
axes[1].set_xlabel("Number of updates")
axes[1].set_ylabel("Training MSE")
axes[1].set_yscale("log")
fig.tight_layout()
plt.show()

In [ ]:
# Supplied: new observations, with a separate seed from the training sample.
# These observations are used only for evaluation, never in the update loop.
torch.manual_seed(8)
x_val = 2 * torch.rand(200, 1) - 1
y_val = x_val**2 + 0.1 * torch.randn(200, 1)

with torch.no_grad():
    training_mse = ((predict(x) - y)**2).mean().item()
    validation_mse = ((predict(x_val) - y_val)**2).mean().item()
print(f"Training MSE:   {training_mse:.6f}")
print(f"Validation MSE: {validation_mse:.6f}")

**Discuss (b–c).**

1. How do predictions and training loss change? Does the trained network pass
   through every observed point?
2. Which observations were used to change the parameters? Why can training and
   validation MSE differ?
3. Distinguish the noisy observations, the true regression function, the initial
   network, and the trained network in the plot.
4. Would interpolating every noisy observation guarantee good predictions for
   new observations? Explain.

Here the validation sample is a first check on new observations. We are not
using it to choose between many models or tune training settings.

*Write your calculation or observations here.*

## Homework: reuse the same tools

Submit one executed notebook per group by **29 October 2026**, in groups of at
most four. Keep the **Group name:** at the top. Include your short code additions,
plots, and mathematical explanations. Follow the course submission instructions.
After completing all tasks, **restart the kernel and run all cells in order**.
The supplied code may remain in your submission. The optional question is not
required.

### A. Change the regression function

Replace the data model by
\[
Y=|X|+\varepsilon,
\]
with the same input and noise distributions. The cells below repeat the supplied
workflow with fresh variables and a fresh network. You only need to change the
target formula, run the cells, and interpret the output. `torch.abs(x)` takes the
absolute value of each entry.

1. Fill the target formula below and run the supplied training and evaluation.
   Describe where the network fits the function well and where it differs.
2. Prove $|x|=\operatorname{ReLU}(x)+\operatorname{ReLU}(-x)$ and give parameters
   of a width-two network that represent $|x|$ exactly.
3. Why need the network trained on noisy observations not reproduce this exact
   representation? Distinguish what the architecture can represent from what
   the fitting procedure finds.

This homework changes the currently stored parameter tensors. If you return to
Exercise 5 afterward, rerun its initialization, training, and results cells.

In [ ]:
# Supplied fresh sample; fill only the new target formula.
torch.manual_seed(7)
x_abs = 2 * torch.rand(64, 1) - 1
noise_abs = 0.1 * torch.randn(64, 1)
y_abs = ...

In [ ]:
# Supplied: reset and reuse exactly the same training workflow.
torch.manual_seed(11)
width = 16
W1 = torch.randn(width, 1, requires_grad=True)
b1 = torch.randn(width, requires_grad=True)
W2 = torch.zeros(1, width, requires_grad=True)
b2 = torch.zeros(1, requires_grad=True)
optimizer = torch.optim.Adam([W1, b1, W2, b2], lr=0.02)
abs_loss_history = []

for step in range(2000):
    optimizer.zero_grad()
    prediction = predict(x_abs)
    loss = ((prediction - y_abs)**2).mean()
    loss.backward()
    optimizer.step()
    abs_loss_history.append(loss.item())

with torch.no_grad():
    abs_train_mse = ((predict(x_abs) - y_abs)**2).mean().item()
    abs_prediction = predict(grid)
abs_loss_history.append(abs_train_mse)

# Supplied independent evaluation sample.
torch.manual_seed(8)
x_abs_val = 2 * torch.rand(200, 1) - 1
y_abs_val = torch.abs(x_abs_val) + 0.1 * torch.randn(200, 1)
with torch.no_grad():
    abs_val_mse = ((predict(x_abs_val) - y_abs_val)**2).mean().item()
print(f"Absolute-value training MSE:   {abs_train_mse:.6f}")
print(f"Absolute-value validation MSE: {abs_val_mse:.6f}")

In [ ]:
# Supplied homework plots.
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(x_abs[:, 0], y_abs[:, 0], s=18, alpha=0.7, label="Training observations")
axes[0].plot(grid[:, 0], torch.abs(grid[:, 0]), "k", label="True function: |x|")
axes[0].plot(grid[:, 0], abs_prediction[:, 0], label="Trained network")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y / prediction")
axes[0].legend()
axes[1].plot(abs_loss_history)
axes[1].set_xlabel("Number of updates")
axes[1].set_ylabel("Training MSE")
axes[1].set_yscale("log")
fig.tight_layout()
plt.show()

*Write your calculation or observations here.*

### B. Population risk and empirical error

Return to the original model $Y=X^2+\varepsilon$. For a fixed measurable
predictor $f$ with finite squared risk, show that
\[
R(f)=\mathbb E[(f(X)-Y)^2]
=0.01+\mathbb E[(f(X)-X^2)^2].
\]
Use independence and the zero mean of $\varepsilon$.

1. Identify the optimal predictor and its risk.
2. Explain why a mean squared error calculated on finitely many observations
   can be below $0.01$, although the population risk cannot.

No new programming is needed.

*Write your calculation or observations here.*

### Optional mathematical question: the best affine predictor

For the original model, find the population-optimal affine predictor $ax+b$
and its risk. You may use
\[
\mathbb E[X]=\mathbb E[X^3]=0,\quad
\mathbb E[X^2]=\frac13,\quad \mathbb E[X^4]=\frac15.
\]
Explain why this model class cannot recover the curved regression function.
No additional programming is required.

*Write your calculation or observations here.*

## Before submitting

- Enter your group name and the group members according to the course instructions.
- Replace the placeholders and add your written answers. Include the plots you discuss.
- Restore the original settings after the in-class experiments: noise standard
  deviation $0.1$ and `h = torch.relu(z)` in `predict`.
- Restart the kernel and run the completed notebook from top to bottom. Check
  that every cell runs and that your conclusions agree with the resulting output.
- Save the executed notebook and submit it through the course submission route.